In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# QUICK-PDE : une fonction Qiskit par ColibriTD
*Consulte la [référence API](https://docs.quantum.ibm.com/api/functions/colibritd-pde)*

> **Note:** Les fonctions Qiskit sont une fonctionnalité expérimentale disponible pour les utilisateurs des plans IBM Quantum&reg; Premium, Flex et On-Prem (via l'API IBM Quantum Platform). Elles sont en version préliminaire et susceptibles de changer.

## Vue d'ensemble
Le solveur d'équations aux dérivées partielles (EDP) présenté ici fait partie de notre plateforme Quantum Innovative Computing Kit (QUICK) (QUICK-PDE), et est packagé comme une fonction Qiskit. Avec la fonction QUICK-PDE, tu peux résoudre des équations aux dérivées partielles spécifiques à un domaine sur des QPU IBM Quantum. Cette fonction est basée sur l'algorithme décrit dans [le document de description H-DES de ColibriTD](https://arxiv.org/abs/2410.01130). Cet algorithme peut résoudre des problèmes multi-physiques complexes, en commençant par la mécanique des fluides numérique (CFD) et la déformation des matériaux (MD), et d'autres cas d'utilisation à venir.

Pour traiter les équations différentielles, les solutions d'essai sont encodées comme des combinaisons linéaires de fonctions orthogonales (généralement des polynômes de Chebyshev, et plus spécifiquement $2^n$ d'entre eux où $n$ est le nombre de qubits encodant ta fonction), paramétrées par les angles d'un circuit quantique variable (VQC). L'ansatz génère un état encodant la fonction, qui est évalué par des observables dont les combinaisons permettent d'évaluer la fonction en tous les points. Tu peux ensuite évaluer la fonction de perte dans laquelle les équations différentielles sont encodées, et affiner les angles dans une boucle hybride, comme indiqué ci-après. Les solutions d'essai se rapprochent progressivement des solutions réelles jusqu'à ce que tu obtiennes un résultat satisfaisant.

![Flux de travail de la fonction QUICK-PDE](../docs/images/guides/colibritd-equation-solver/diagram.svg)

En plus de cette boucle hybride, tu peux également enchaîner différents optimiseurs. C'est utile lorsque tu veux qu'un optimiseur global trouve un bon ensemble d'angles, puis un optimiseur plus fin pour suivre un gradient vers le meilleur ensemble d'angles voisins. Dans le cas de la mécanique des fluides numérique (CFD), la séquence d'optimisation par défaut produit les meilleurs résultats - mais dans le cas de la déformation des matériaux (MD), bien que la valeur par défaut donne de bons résultats, tu peux la configurer davantage pour des avantages spécifiques au problème.

Note que pour chaque variable de la fonction, tu spécifies le nombre de qubits (avec lequel tu peux jouer). En empilant 10 circuits identiques et en évaluant les 10 observables identiques sur différents qubits tout au long d'un grand circuit, tu peux atténuer le bruit dans le processus d'optimisation CMA, en t'appuyant sur la méthode d'apprentissage du bruit, et réduire significativement le nombre de shots nécessaires.

### Mécanique des fluides numérique
L'équation de Burgers non visqueuse modélise les fluides non visqueux en écoulement comme suit :

$$\frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x} = 0,$$

$u$ représente le champ de vitesse du fluide. Ce cas d'utilisation a une condition aux limites temporelle : tu peux sélectionner la condition initiale puis laisser le système se relaxer. Actuellement, les seules conditions initiales acceptées sont les fonctions linéaires : $ax + b$.

Les arguments pour les équations différentielles de la CFD sont sur une grille fixe, comme suit :

- $t$ est compris entre 0 et 0,95 avec 30 points d'échantillonnage. $x$ est compris entre 0 et 0,95 avec un pas de 0,2375.

### Déformation des matériaux
Ce cas d'utilisation se concentre sur la déformation hypoélastique avec le test de traction unidimensionnel, dans lequel une barre fixée dans l'espace est tirée à son autre extrémité. Nous décrivons le problème comme suit :

$$u' - \frac{\sigma}{3K} - \frac{2}{\sqrt{3}}\epsilon_0\left(\frac{\sigma'}{\sigma_0\sqrt{3}}\right)^n = 0$$

$$\sigma' - b = 0,$$

$K$ représente le module de compressibilité du matériau étiré, $n$ l'exposant d'une loi de puissance, $b$ la force par unité de masse, $\epsilon_0$ la limite de contrainte proportionnelle, $\sigma_0$ la limite de déformation proportionnelle, $u$ la fonction de contrainte et $\sigma$ la fonction de déformation.

La barre considérée est de longueur unitaire. Ce cas d'utilisation a une condition aux limites pour la contrainte de surface $t$, ou la quantité de travail nécessaire pour étirer la barre.

Les arguments pour les équations différentielles de la MD sont sur une grille fixe, comme suit :

- $x$ est compris entre 0 et 1 avec un pas de 0,04.

## Benchmarks
Le tableau suivant présente des statistiques sur différentes exécutions de notre fonction.

| Exemple                            | Nombre de qubits | Initialisation        | Erreur    | Temps total (min) | Utilisation runtime (min) |
| ---------------------------------- | ---------------- | --------------------- | --------- | ----------------- | ------------------------- |
| Équation de Burgers non visqueuse  | 50               | `PHYSICALLY_INFORMED` | $10^{-2}$ | 66                | 25                        |
| Test de traction hypoélastique 1D  | 18               | `RANDOM`              | $10^{-2}$ | 123               | 100                       |

## Premiers pas
Remplis le [formulaire pour demander l'accès à la fonction QUICK-PDE](https://forms.cloud.microsoft/e/3Wi9cbjQPK). Ensuite, en supposant que tu as déjà [enregistré ton compte](/guides/functions#install-qiskit-functions-catalog-client) dans ton environnement local, sélectionne la fonction comme suit :

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

quick = catalog.load("colibritd/quick-pde")

## Exemples

Pour démarrer, essaie l'un des exemples suivants :

### Mécanique des fluides numérique (CFD)

Lorsque les conditions initiales sont définies à $u(0,x) = x$, les résultats sont les suivants :

In [ ]:
# launch the simulation with initial conditions u(0,x) = a*x + b
job = quick.run(use_case="cfd", physical_parameters={"a": 1.0, "b": 0.0})

Vérifie le [statut](/guides/functions#check-job-status) de ta charge de travail Qiskit Function ou récupère les [résultats](/guides/functions#retrieve-results) comme suit :

In [ ]:
print(job.status())
solution = job.result()

'QUEUED'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

_ = plt.figure()
ax = plt.axes(projection="3d")

# plot the solution using the 3d plotting capabilities of pyplot
t, x = np.meshgrid(solution["samples"]["t"], solution["samples"]["x"])
ax.plot_surface(
    t,
    x,
    solution["functions"]["u"],
    edgecolor="royalblue",
    lw=0.25,
    rstride=26,
    cstride=26,
    alpha=0.3,
)
ax.scatter(t, x, solution["functions"]["u"], marker=".")
ax.set(xlabel="t", ylabel="x", zlabel="u(t,x)")

plt.show()

<Image src="../docs/images/guides/colibritd-pde/extracted-outputs/c42aba9b-0.avif" alt="Output of the previous code cell" />

### Material deformation

The material deformation use case requires the physical parameters of your material and the applied force, as follows:

In [ ]:
import matplotlib.pyplot as plt

# select the properties of your material
job = quick.run(
    use_case="md",
    physical_parameters={
        "t": 12.0,
        "K": 100.0,
        "n": 4.0,
        "b": 10.0,
        "epsilon_0": 0.1,
        "sigma_0": 5.0,
    },
)

# plot the result
solution = job.result()

_ = plt.figure()
stress_plot = plt.subplot(211)
plt.plot(solution["samples"]["x"], solution["functions"]["u"])
strain_plot = plt.subplot(212)
plt.plot(solution["samples"]["x"], solution["functions"]["sigma"])

plt.show()

<Image src="../docs/images/guides/colibritd-pde/extracted-outputs/a568e325-0.avif" alt="Output of the previous code cell" />

![Sortie de la cellule de code précédente](../docs/images/guides/colibritd-pde/extracted-outputs/c42aba9b-0.avif)

### Déformation des matériaux
Le cas d'utilisation de déformation des matériaux nécessite les paramètres physiques de ton matériau et la force appliquée, comme suit :

In [ ]:
# u(t=0.2, x=0.7) == 2
assert solution["samples"]["t"][1] == 0.2
assert solution["samples"]["x"][2] == 0.7
assert solution["functions"]["u"][1, 2] == 2

![Sortie de la cellule de code précédente](../docs/images/guides/colibritd-pde/extracted-outputs/a568e325-0.avif)

Voici un exemple de comment obtenir la valeur de la fonction pour un ensemble spécifique de coordonnées :

In [ ]:
job = quick.run(use_case="mdf", physical_params={})

print(job.error_message())


# or write a wrapper around it for a more human readable version
def pprint_error(job):
    print("".join(eval(job.error_message())["error"]))


print("___")
pprint_error(job)

{"error": ["qiskit.exceptions.QiskitError: 'Unknown argument \"physical_params\", did you make a typo? -- https://docs.quantum.ibm.com/errors#1804'\n"]}
___
qiskit.exceptions.QiskitError: 'Unknown argument "physical_params", did you make a typo? -- https://docs.quantum.ibm.com/errors#1804'



## Récupérer les messages d'erreur
Si le statut de ta charge de travail est `ERROR`, utilise `job.error_message()` pour récupérer le message d'erreur et aider au débogage, comme suit :